# Section 1: Introduction

Notebook này gom toàn bộ quy trình cho đề tài phân loại ảnh phương tiện giao thông: mô tả dataset, làm sạch dữ liệu, chia train/validation/test, tiền xử lý ảnh, trực quan hóa đặc trưng, train MobileNet và ResNet từ đầu, đánh giá và thảo luận kết quả.

**Danh sách class mục tiêu:** `bicycle`, `boat`, `bus`, `car`, `helicopter`, `minibus`, `motorcycle`, `taxi`, `train`, `truck`.

**Mục tiêu dữ liệu:** hơn 10.000 ảnh sau khi crawl và làm sạch, mỗi class có số lượng mẫu tương đối đủ để tránh mất cân bằng quá lớn.

**Ràng buộc mô hình:** không dùng pretrained weights, không transfer learning, không fine-tuning. MobileNet và ResNet trong notebook này được xây dựng bằng các block CNN từ đầu và khởi tạo trọng số ngẫu nhiên.

## Section 2: Environment Setup

Cell dưới đây import thư viện, kiểm tra GPU, cấu hình đường dẫn dataset và các siêu tham số huấn luyện. Notebook tự tìm dataset ở các vị trí phổ biến của Kaggle/Colab/local; nếu cần, chỉ việc sửa biến `DATA_ROOT`.

In [ ]:
from pathlib import Path
import os
import random
import shutil
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

try:
    import imagehash
except ImportError:
    imagehash = None
    print('imagehash is not installed. pHash/dHash demo cells will be skipped unless you install it.')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

REQUIRED_CLASSES = ['bicycle', 'boat', 'bus', 'car', 'helicopter', 'minibus', 'motorcycle', 'taxi', 'train', 'truck']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RUN_TRAINING = True
EXPORT_SPLIT_FOLDERS = False  # set True if you want to materialize data/splits folders from this notebook
MAX_IMAGES_PER_CLASS = None  # set a small number, e.g. 200, for a quick smoke test

def find_dataset_root():
    env_path = os.environ.get('DATA_ROOT')
    candidates = []
    if env_path:
        candidates.append(Path(env_path))

    candidates.extend([
        Path('/kaggle/input/traffic-vehicle-classification/data/cleaned'),
        Path('/kaggle/input/traffic-vehicle-classification/cleaned'),
        Path('/content/traffic_vehicle_classification/data/cleaned'),
        Path('/content/data/cleaned'),
        Path('../data/cleaned'),
        Path('data/cleaned'),
    ])

    for pattern in ['/kaggle/input/*/data/cleaned', '/kaggle/input/*/cleaned']:
        candidates.extend(Path('/').glob(pattern.lstrip('/')))

    for path in candidates:
        if path.exists() and any(child.is_dir() for child in path.iterdir()):
            return path
    return candidates[0]

DATA_ROOT = find_dataset_root()
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Section 3: Dataset Loading

Dataset được đọc từ folder dạng `data/cleaned/<class_name>/*.jpg`. Notebook tự lấy tên class từ tên thư mục và tạo dataframe gồm `image_path`, `label`, `width`, `height`, `channel`, `file_size`.

In [ ]:
def get_image_info(path):
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)
            width, height = img.size
            channel = len(img.getbands())
            return {
                'width': width,
                'height': height,
                'channel': channel,
                'mode': img.mode,
                'format': img.format,
                'file_size': path.stat().st_size,
                'file_size_kb': path.stat().st_size / 1024,
                'error': None,
            }
    except Exception as exc:
        return {
            'width': np.nan,
            'height': np.nan,
            'channel': np.nan,
            'mode': None,
            'format': None,
            'file_size': path.stat().st_size if path.exists() else np.nan,
            'file_size_kb': path.stat().st_size / 1024 if path.exists() else np.nan,
            'error': str(exc),
        }

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DATA_ROOT}. Please update DATA_ROOT in Section 2.')

class_names = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])
print('Detected classes:', class_names)

missing_required = sorted(set(REQUIRED_CLASSES) - set(class_names))
if missing_required:
    print('Warning - missing required classes:', missing_required)

rows = []
for label in class_names:
    image_paths = [p for p in sorted((DATA_ROOT / label).rglob('*')) if p.suffix.lower() in IMAGE_EXTENSIONS]
    if MAX_IMAGES_PER_CLASS is not None:
        image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
    for path in image_paths:
        info = get_image_info(path)
        rows.append({'image_path': str(path), 'label': label, **info})

df_all = pd.DataFrame(rows)
if df_all.empty:
    raise ValueError('No images found. Check DATA_ROOT and folder structure.')

error_df = df_all[df_all['error'].notna()].copy()
df = df_all[df_all['error'].isna()].copy().reset_index(drop=True)
df['aspect_ratio'] = df['width'] / df['height']

print('Total files scanned:', len(df_all))
print('Valid images:', len(df))
print('Unreadable/error images:', len(error_df))
display(df.head())

## Section 4: Dataset Overview and Statistics

Phần này thống kê tổng số mẫu, số mẫu từng class, kiểu dữ liệu, số mẫu lỗi/trống, kích thước ảnh và trực quan hóa bằng bar chart, histogram, boxplot.

In [ ]:
print('Total valid samples:', len(df))
print('Number of classes:', df['label'].nunique())

class_counts = df['label'].value_counts().sort_index()
display(class_counts.to_frame('count'))

print('Data types:')
display(df.dtypes.to_frame('dtype'))

print('Missing values per column:')
display(df.isna().sum().to_frame('missing_count'))

size_summary = df[['width', 'height', 'channel', 'file_size_kb', 'aspect_ratio']].describe().T
display(size_summary.round(2))

label_size_summary = df.groupby('label').agg(
    count=('image_path', 'count'),
    width_min=('width', 'min'), width_max=('width', 'max'), width_mean=('width', 'mean'), width_median=('width', 'median'),
    height_min=('height', 'min'), height_max=('height', 'max'), height_mean=('height', 'mean'), height_median=('height', 'median'),
    file_size_kb_mean=('file_size_kb', 'mean'),
).reset_index()
display(label_size_summary.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=df, x='label', order=class_counts.index, ax=axes[0, 0])
axes[0, 0].set_title('Class distribution')
axes[0, 0].tick_params(axis='x', rotation=35)

sns.histplot(data=df, x='width', bins=40, kde=True, ax=axes[0, 1], color='#2f6f9f')
axes[0, 1].set_title('Width histogram')

sns.histplot(data=df, x='height', bins=40, kde=True, ax=axes[1, 0], color='#3c8d57')
axes[1, 0].set_title('Height histogram')

sns.boxplot(data=df, x='label', y='file_size_kb', order=class_counts.index, ax=axes[1, 1])
axes[1, 1].set_title('File size boxplot by class')
axes[1, 1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=df, x='label', y='width', order=class_counts.index, ax=axes[0])
axes[0].set_title('Width boxplot by class')
axes[0].tick_params(axis='x', rotation=35)

sns.boxplot(data=df, x='label', y='height', order=class_counts.index, ax=axes[1])
axes[1].set_title('Height boxplot by class')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
def show_samples(dataframe, labels, samples_per_class=3):
    sample_rows = []
    for label in labels:
        label_df = dataframe[dataframe['label'] == label]
        if len(label_df) == 0:
            continue
        sample_rows.extend(label_df.sample(min(samples_per_class, len(label_df)), random_state=SEED).to_dict('records'))

    cols = samples_per_class
    rows_n = max(1, int(np.ceil(len(sample_rows) / cols)))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 3 * rows_n))
    axes = np.array(axes).reshape(-1)
    for ax, row in zip(axes, sample_rows):
        img = Image.open(row['image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(row['label'])
        ax.axis('off')
    for ax in axes[len(sample_rows):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(df, class_counts.index.tolist(), samples_per_class=3)

## Section 5: Data Cleaning Summary

Quy trình làm sạch dữ liệu nên được thực hiện sau khi crawl:

1. Mở từng ảnh bằng Pillow/OpenCV để phát hiện ảnh lỗi hoặc ảnh không đọc được.
2. Loại ảnh quá nhỏ, ví dụ nhỏ hơn `128x128`, vì ảnh quá nhỏ thường thiếu chi tiết để phân loại.
3. Chuẩn hóa ảnh về RGB JPEG để giảm lỗi khi load dataset.
4. Đổi tên ảnh theo format thống nhất, ví dụ `car_000001.jpg`.
5. Lọc ảnh trùng hoặc gần trùng bằng pHash/dHash. Hai ảnh có hash gần nhau theo Hamming distance nhỏ hơn ngưỡng, ví dụ `<= 6`, được xem là trùng hoặc gần trùng.

Công thức Hamming distance: số bit khác nhau giữa hai chuỗi hash. Với `imagehash`, có thể tính trực tiếp bằng phép trừ: `distance = hash_1 - hash_2`.

In [ ]:
def count_folder_images(root):
    root = Path(root)
    rows = []
    if not root.exists():
        return pd.DataFrame(columns=['label', 'count'])
    for label_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        count = sum(1 for p in label_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
        rows.append({'label': label_dir.name, 'count': count})
    return pd.DataFrame(rows)

raw_root = DATA_ROOT.parent / 'raw'
cleaned_root = DATA_ROOT
rejected_root = DATA_ROOT.parent / 'rejected'
duplicates_root = DATA_ROOT.parent / 'duplicates'

raw_counts = count_folder_images(raw_root).rename(columns={'count': 'raw_count'})
clean_counts = count_folder_images(cleaned_root).rename(columns={'count': 'cleaned_count'})
rejected_counts = count_folder_images(rejected_root).rename(columns={'count': 'rejected_count'})
duplicate_counts = count_folder_images(duplicates_root).rename(columns={'count': 'duplicate_count'})

cleaning_summary = clean_counts.merge(raw_counts, on='label', how='left')
cleaning_summary = cleaning_summary.merge(rejected_counts, on='label', how='left')
cleaning_summary = cleaning_summary.merge(duplicate_counts, on='label', how='left')
cleaning_summary = cleaning_summary.fillna(0)
display(cleaning_summary)

print('If raw/rejected/duplicates folders are unavailable on Kaggle, the table still reports cleaned_count from DATA_ROOT.')

In [ ]:
def compute_image_hash(path, method='phash'):
    if imagehash is None:
        return None
    with Image.open(path) as img:
        if method == 'phash':
            return imagehash.phash(img)
        if method == 'dhash':
            return imagehash.dhash(img)
        raise ValueError('method must be phash or dhash')

sample_paths = df['image_path'].head(2).tolist()
if imagehash is not None and len(sample_paths) == 2:
    h1 = compute_image_hash(sample_paths[0], 'phash')
    h2 = compute_image_hash(sample_paths[1], 'phash')
    print('pHash image 1:', h1)
    print('pHash image 2:', h2)
    print('Hamming distance:', h1 - h2)
else:
    print('Skip hash demo because imagehash is not installed or not enough images.')

In [ ]:
def show_rejected_or_duplicate_examples(root, title, max_images=8):
    root = Path(root)
    paths = []
    if root.exists():
        paths = [p for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS][:max_images]
    if not paths:
        print(f'No example images found for {title}: {root}')
        return
    cols = min(4, len(paths))
    rows_n = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 3 * rows_n))
    axes = np.array(axes).reshape(-1)
    for ax, path in zip(axes, paths):
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(path.parent.name)
        ax.axis('off')
    for ax in axes[len(paths):]:
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_rejected_or_duplicate_examples(rejected_root, 'Rejected images')
show_rejected_or_duplicate_examples(duplicates_root, 'Duplicate images')

## Section 6: Train / Validation / Test Split

Dữ liệu được chia theo tỉ lệ `70/15/15` bằng stratified split theo label. Stratified split giúp giữ phân bố class giữa train, validation và test gần giống nhau.

In [ ]:
if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-6:
    raise ValueError('TRAIN_RATIO + VAL_RATIO + TEST_RATIO must equal 1.0')

min_class_count = df['label'].value_counts().min()
if min_class_count < 3:
    raise ValueError('Each class needs at least 3 images for train/val/test stratified split.')

train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_RATIO,
    stratify=df['label'],
    random_state=SEED,
)
val_relative_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative_ratio,
    stratify=train_val_df['label'],
    random_state=SEED,
)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df = val_df.copy(); val_df['split'] = 'val'
test_df = test_df.copy(); test_df['split'] = 'test'
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_df.to_csv(OUTPUT_DIR / 'split_dataframe.csv', index=False)

print('Train:', len(train_df), 'Validation:', len(val_df), 'Test:', len(test_df))
display(split_df.groupby(['split', 'label']).size().unstack(fill_value=0))

In [ ]:
def export_split_folders(split_dataframe, output_root):
    output_root = Path(output_root)
    for _, row in split_dataframe.iterrows():
        src = Path(row['image_path'])
        dest = output_root / row['split'] / row['label'] / src.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            shutil.copy2(src, dest)

if EXPORT_SPLIT_FOLDERS:
    split_output = OUTPUT_DIR / 'data' / 'splits'
    export_split_folders(split_df, split_output)
    print('Split folders exported to:', split_output)
else:
    print('EXPORT_SPLIT_FOLDERS=False, split is stored as dataframe CSV only.')

In [ ]:
split_counts = split_df.groupby(['split', 'label']).size().reset_index(name='count')
plt.figure(figsize=(14, 5))
sns.barplot(data=split_counts, x='label', y='count', hue='split')
plt.title('Class distribution across train/validation/test')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

distribution = pd.crosstab(split_df['label'], split_df['split'], normalize='columns')
distribution['train_test_abs_diff'] = (distribution['train'] - distribution['test']).abs()
display(distribution.round(4))

max_shift = distribution['train_test_abs_diff'].max()
if max_shift < 0.02:
    print(f'Distribution shift is small. Max train-test class proportion difference = {max_shift:.4f}')
else:
    print(f'Potential distribution shift. Max train-test class proportion difference = {max_shift:.4f}')

## Section 7: Image Preprocessing

Ảnh được resize về `224x224` để tạo batch tensor có cùng kích thước. Pixel được normalize về `[0, 1]` để mô hình học ổn định hơn. Data augmentation chỉ áp dụng trên train set, không áp dụng cho validation/test.

In [ ]:
label_to_id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(label_to_id)

for frame in [train_df, val_df, test_df, split_df]:
    frame['label_id'] = frame['label'].map(label_to_id)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.10),
], name='train_augmentation')

def load_and_preprocess(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(dataframe, training=False):
    paths = dataframe['image_path'].astype(str).values
    labels = dataframe['label_id'].astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print('Number of classes:', num_classes)
print(label_to_id)

In [ ]:
sample_row = train_df.sample(1, random_state=SEED).iloc[0]
original = Image.open(sample_row['image_path']).convert('RGB')
resized = original.resize(IMG_SIZE)
normalized = np.asarray(resized).astype('float32') / 255.0

augmented_batch = data_augmentation(tf.expand_dims(normalized, axis=0), training=True)
augmented = augmented_batch[0].numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original); axes[0].set_title(f'Original {original.size}')
axes[1].imshow(resized); axes[1].set_title('Resized 224x224')
axes[2].hist(normalized.ravel(), bins=40); axes[2].set_title('Normalized pixel values')
axes[3].imshow(np.clip(augmented, 0, 1)); axes[3].set_title('Train augmentation example')
for ax in [axes[0], axes[1], axes[3]]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Section 8: Feature Extraction and Visualization

CNN sẽ học đặc trưng trực tiếp từ pixel ảnh đã resize/normalize. Để trực quan hóa trước khi train, ta dùng pixel feature đơn giản từ ảnh downsample `32x32` kết hợp PCA/t-SNE. Biểu đồ 2D giúp quan sát sơ bộ class nào dễ tách và class nào có thể chồng lấn.

In [ ]:
def build_visualization_features(dataframe, max_per_class=80):
    sampled = []
    for label in sorted(dataframe['label'].unique()):
        part = dataframe[dataframe['label'] == label]
        sampled.append(part.sample(min(max_per_class, len(part)), random_state=SEED))
    sample_df = pd.concat(sampled, ignore_index=True)

    features_raw = []
    features_norm = []
    labels = []
    for _, row in sample_df.iterrows():
        img = Image.open(row['image_path']).convert('RGB').resize((32, 32))
        arr = np.asarray(img).astype('float32')
        features_raw.append(arr.ravel())
        features_norm.append((arr / 255.0).ravel())
        labels.append(row['label'])
    return np.asarray(features_raw), np.asarray(features_norm), np.asarray(labels)

X_raw, X_norm, y_vis = build_visualization_features(df, max_per_class=80)
print('Raw pixel feature shape:', X_raw.shape)
print('Normalized pixel feature shape:', X_norm.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_raw.ravel(), bins=50, color='#756bb1')
axes[0].set_title('Pixel distribution before normalization')
axes[1].hist(X_norm.ravel(), bins=50, color='#238b45')
axes[1].set_title('Pixel distribution after normalization')
plt.tight_layout()
plt.show()

In [ ]:
X_scaled = StandardScaler().fit_transform(X_norm)
pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({'pc1': pca_coords[:, 0], 'pc2': pca_coords[:, 1], 'label': y_vis})

plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x='pc1', y='pc2', hue='label', s=35, alpha=0.85)
plt.title('PCA visualization of normalized pixel features')
plt.tight_layout()
plt.show()

print('PCA explained variance ratio:', pca.explained_variance_ratio_)

In [ ]:
if len(X_scaled) >= 50:
    n_vis = min(len(X_scaled), 800)
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_scaled), size=n_vis, replace=False)
    perplexity = min(30, max(5, n_vis // 10))
    tsne = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=SEED)
    tsne_coords = tsne.fit_transform(X_scaled[idx])
    tsne_df = pd.DataFrame({'x': tsne_coords[:, 0], 'y': tsne_coords[:, 1], 'label': y_vis[idx]})
    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=tsne_df, x='x', y='y', hue='label', s=35, alpha=0.85)
    plt.title('t-SNE visualization of normalized pixel features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough samples for t-SNE visualization.')

## Section 9: Model 1 - MobileNet From Scratch

MobileNet dùng **depthwise separable convolution**: depthwise convolution học bộ lọc riêng cho từng channel, sau đó pointwise convolution `1x1` trộn thông tin giữa các channel. Cách này giảm số lượng tham số so với convolution thường.

Kiến trúc trong notebook:

- Input size: `224x224x3`.
- Các convolution block và separable convolution block.
- Batch normalization sau convolution.
- Activation: ReLU.
- Global average pooling.
- Dropout.
- Dense output softmax.

Mô hình được khởi tạo ngẫu nhiên, không dùng `weights='imagenet'` hoặc bất kỳ pretrained model nào.

In [ ]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1, name=None):
    x = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False, name=None if name is None else name + '_conv')(x)
    x = tf.keras.layers.BatchNormalization(name=None if name is None else name + '_bn')(x)
    x = tf.keras.layers.ReLU(name=None if name is None else name + '_relu')(x)
    return x

def depthwise_separable_block(x, filters, strides=1, name='ds'):
    x = tf.keras.layers.DepthwiseConv2D(3, strides=strides, padding='same', use_bias=False, name=name + '_depthwise')(x)
    x = tf.keras.layers.BatchNormalization(name=name + '_dw_bn')(x)
    x = tf.keras.layers.ReLU(name=name + '_dw_relu')(x)
    x = tf.keras.layers.Conv2D(filters, 1, padding='same', use_bias=False, name=name + '_pointwise')(x)
    x = tf.keras.layers.BatchNormalization(name=name + '_pw_bn')(x)
    x = tf.keras.layers.ReLU(name=name + '_pw_relu')(x)
    return x

def build_mobilenet_from_scratch(num_classes):
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = conv_bn_relu(inputs, 32, strides=2, name='stem')
    x = depthwise_separable_block(x, 64, strides=1, name='ds_1')
    x = depthwise_separable_block(x, 128, strides=2, name='ds_2')
    x = depthwise_separable_block(x, 128, strides=1, name='ds_3')
    x = depthwise_separable_block(x, 256, strides=2, name='ds_4')
    x = depthwise_separable_block(x, 256, strides=1, name='ds_5')
    x = depthwise_separable_block(x, 512, strides=2, name='ds_6')
    for i in range(3):
        x = depthwise_separable_block(x, 512, strides=1, name=f'ds_6_repeat_{i + 1}')
    x = depthwise_separable_block(x, 1024, strides=2, name='ds_7')
    x = depthwise_separable_block(x, 1024, strides=1, name='ds_8')
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(0.35, name='dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier')(x)
    model = tf.keras.Model(inputs, outputs, name='MobileNet_From_Scratch')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

mobilenet_model = build_mobilenet_from_scratch(num_classes)
mobilenet_model.summary()

In [ ]:
callbacks_mobilenet = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
]

mobilenet_history = None
mobilenet_train_time = 0
if RUN_TRAINING:
    start = time.time()
    mobilenet_history = mobilenet_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_mobilenet,
    )
    mobilenet_train_time = time.time() - start
else:
    print('RUN_TRAINING=False, skip MobileNet training.')

In [ ]:
def plot_training_history(history, title):
    if history is None:
        print(f'No history for {title}.')
        return
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist['loss'], label='train_loss')
    axes[0].plot(hist['val_loss'], label='val_loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[1].plot(hist['accuracy'], label='train_accuracy')
    axes[1].plot(hist['val_accuracy'], label='val_accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_training_history(mobilenet_history, 'MobileNet From Scratch')

## Section 10: Model 2 - ResNet From Scratch

ResNet dùng **residual block** và **skip connection**. Thay vì buộc các lớp học trực tiếp một ánh xạ phức tạp, residual block học phần sai khác so với input. Skip connection giúp gradient truyền tốt hơn qua mạng sâu, giảm hiện tượng vanishing gradient.

Kiến trúc trong notebook:

- Input size: `224x224x3`.
- Stem convolution + max pooling.
- Các residual stages với filter tăng dần.
- Batch normalization.
- Activation: ReLU.
- Global average pooling.
- Dropout.
- Dense output softmax.

Mô hình được xây từ các layer Keras cơ bản và khởi tạo ngẫu nhiên.

In [ ]:
def residual_block(x, filters, strides=1, name='res'):
    shortcut = x

    x = tf.keras.layers.Conv2D(filters, 3, strides=strides, padding='same', use_bias=False, name=name + '_conv1')(x)
    x = tf.keras.layers.BatchNormalization(name=name + '_bn1')(x)
    x = tf.keras.layers.ReLU(name=name + '_relu1')(x)
    x = tf.keras.layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False, name=name + '_conv2')(x)
    x = tf.keras.layers.BatchNormalization(name=name + '_bn2')(x)

    if shortcut.shape[-1] != filters or strides != 1:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=strides, padding='same', use_bias=False, name=name + '_shortcut_conv')(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=name + '_shortcut_bn')(shortcut)

    x = tf.keras.layers.Add(name=name + '_add')([x, shortcut])
    x = tf.keras.layers.ReLU(name=name + '_out_relu')(x)
    return x

def build_resnet_from_scratch(num_classes):
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = conv_bn_relu(inputs, 64, kernel_size=7, strides=2, name='stem')
    x = tf.keras.layers.MaxPooling2D(pool_size=3, strides=2, padding='same', name='stem_pool')(x)

    block_id = 1
    for stage, filters in enumerate([64, 128, 256, 512], start=1):
        for block in range(2):
            strides = 2 if block == 0 and stage > 1 else 1
            x = residual_block(x, filters, strides=strides, name=f'stage{stage}_block{block + 1}')
            block_id += 1

    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(0.4, name='dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier')(x)
    model = tf.keras.Model(inputs, outputs, name='ResNet_From_Scratch')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

resnet_model = build_resnet_from_scratch(num_classes)
resnet_model.summary()

In [ ]:
callbacks_resnet = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
]

resnet_history = None
resnet_train_time = 0
if RUN_TRAINING:
    start = time.time()
    resnet_history = resnet_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_resnet,
    )
    resnet_train_time = time.time() - start
else:
    print('RUN_TRAINING=False, skip ResNet training.')

plot_training_history(resnet_history, 'ResNet From Scratch')

## Section 11: Model Evaluation

Đánh giá hai mô hình trên test set bằng Accuracy, Precision, Recall, F1-score, confusion matrix và classification report. Bảng so sánh cũng ghi số tham số và thời gian train.

In [ ]:
def collect_predictions(model, dataset):
    y_true = []
    y_pred = []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(np.argmax(probs, axis=1).tolist())
    return np.array(y_true), np.array(y_pred)

def evaluate_model(model, model_name, training_time):
    y_true, y_pred = collect_predictions(model, test_ds)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    print('\n' + '=' * 80)
    print(model_name)
    print('=' * 80)
    print(classification_report(y_true, y_pred, target_names=[id_to_label[i] for i in range(num_classes)], zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[id_to_label[i] for i in range(num_classes)], yticklabels=[id_to_label[i] for i in range(num_classes)])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.xticks(rotation=35, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return {
        'Model': model_name,
        'Test Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'Number of parameters': model.count_params(),
        'Training time (seconds)': training_time,
    }, cm

evaluation_rows = []
confusion_matrices = {}

if RUN_TRAINING:
    row, cm = evaluate_model(mobilenet_model, 'MobileNet From Scratch', mobilenet_train_time)
    evaluation_rows.append(row)
    confusion_matrices['MobileNet From Scratch'] = cm

    row, cm = evaluate_model(resnet_model, 'ResNet From Scratch', resnet_train_time)
    evaluation_rows.append(row)
    confusion_matrices['ResNet From Scratch'] = cm
else:
    print('RUN_TRAINING=False, evaluation is skipped.')

results_df = pd.DataFrame(evaluation_rows)
if not results_df.empty:
    display(results_df.round(4))
    results_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

In [ ]:
if not results_df.empty:
    metric_cols = ['Test Accuracy', 'Precision', 'Recall', 'F1-score']
    plot_df = results_df.melt(id_vars='Model', value_vars=metric_cols, var_name='Metric', value_name='Score')
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model')
    plt.ylim(0, 1)
    plt.title('MobileNet vs ResNet metrics on test set')
    plt.tight_layout()
    plt.show()

    params_time_df = results_df[['Model', 'Number of parameters', 'Training time (seconds)']].copy()
    display(params_time_df)

## Section 12: Result Discussion

Sau khi chạy training/evaluation, hãy dựa vào bảng metric và confusion matrix để nhận xét:

- Mô hình có `F1-score` và `Test Accuracy` cao hơn thường là mô hình tổng quát tốt hơn trên test set.
- MobileNet thường nhẹ hơn vì dùng depthwise separable convolution, phù hợp khi cần tốc độ và ít tham số.
- ResNet có skip connection nên có thể học biểu diễn sâu hơn, nhưng cũng có nhiều tham số hơn và cần dữ liệu sạch để tránh overfit.
- Các class dễ nhầm lẫn thường là `taxi` và `car`, `minibus` và `bus`, `truck` và `bus`, vì hình dáng hoặc góc chụp có thể giống nhau.
- Chất lượng dữ liệu crawl ảnh hưởng rất lớn: ảnh sai nhãn, ảnh quá nhỏ, watermark, ảnh cắt mất phương tiện hoặc ảnh nhiều đối tượng đều làm mô hình học nhiễu.
- Nếu class mất cân bằng, mô hình có xu hướng ưu tiên class nhiều mẫu. Khi đó nên crawl thêm class thiếu, lọc thủ công kỹ hơn hoặc dùng augmentation hợp lý.
- Mục tiêu `accuracy >= 85%` khả thi hơn khi mỗi class có đủ ảnh sạch, ít trùng, split cân bằng và train đủ epoch.

In [ ]:
if not results_df.empty:
    best_row = results_df.sort_values('F1-score', ascending=False).iloc[0]
    print(f"Best model by macro F1-score: {best_row['Model']}")
    print(f"Test Accuracy: {best_row['Test Accuracy']:.4f}")
    print(f"F1-score: {best_row['F1-score']:.4f}")
    if best_row['Test Accuracy'] >= 0.85:
        print('The model reaches the target accuracy >= 85%.')
    else:
        print('The model has not reached 85% accuracy yet. Improve data quality, class balance, augmentation, epochs, or learning rate.')
else:
    print('Run training first to generate automatic result discussion.')

## Section 13: Conclusion

Notebook đã trình bày đầy đủ quy trình cho đề tài:

1. Crawl data theo class phương tiện giao thông.
2. Clean data: xóa ảnh lỗi, ảnh quá nhỏ, chuẩn hóa RGB/JPEG, đổi tên thống nhất.
3. Deduplicate data bằng pHash/dHash và Hamming distance.
4. Chia train/validation/test bằng stratified split.
5. Preprocess ảnh: resize `224x224`, normalize pixel `[0, 1]`, augmentation cho train set.
6. Trực quan hóa đặc trưng bằng PCA/t-SNE.
7. Train MobileNet từ đầu.
8. Train ResNet từ đầu.
9. Đánh giá bằng Accuracy, Precision, Recall, F1-score, classification report và confusion matrix.

**Hạn chế:** dữ liệu crawl có thể nhiễu, một số class giống nhau về hình dạng, model train từ đầu cần nhiều dữ liệu và thời gian hơn so với transfer learning.

**Hướng cải thiện:** crawl thêm dữ liệu, lọc thủ công kỹ hơn, cân bằng class, tăng augmentation, train nhiều epoch hơn, thử optimizer hoặc learning rate schedule tốt hơn.